In [6]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128

Found existing installation: torch 2.15.0+cu132
Uninstalling torch-2.15.0+cu132:
  Successfully uninstalled torch-2.15.0+cu132
Found existing installation: torchvision 0.29.2+cu132
Uninstalling torchvision-0.29.2+cu132:
  Successfully uninstalled torchvision-0.29.2+cu132
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


Looking in indexes: https://download.pytorch.org/whl/cu128
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 90.4 MB/s eta 0:00:31
    --------------------------------------- 0.0/2.8 GB 98.1 MB/s eta 0:00:28
    --------------------------------------- 0.1/2.8 GB 104.6 MB/s eta 0:00:26
   - -------------------------------------- 0.1/2.8 GB 110.2 MB/s eta 0:00:25
   - -------------------------------------- 0.1/2.8 GB 111.8 MB/s eta 0:00:24
   - -------------------------------------- 0.1/2.8 GB 112.8 MB/s eta 0:00:24
   -- ------------------------------------- 0.2/2.8 GB 113.4 MB/s eta 0:00:23
   -- ------------------------------------- 0.2/2.8 GB 113.9 MB/s eta 0:00:23
   --- ------------------------------------ 0.2/2.8 GB 114.4 MB/s eta 0:00:23
   --- ------------------------------------ 0.2/2.8 GB 114.7 MB/s eta 0:00:22
   --- ------------------------------------ 0.3/2.8 GB 115.2 MB/s eta 0:00:22
   ---- --------


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No se detectó GPU. Entrenar en CPU está bloqueado. '
        'Revisa driver y que torch esté instalado con build CUDA (NVIDIA) o ROCm (AMD).'
    )

DEVICE = torch.device('cuda:0')

backend = f'ROCm/HIP {torch.version.hip}' if torch.version.hip else f'CUDA {torch.version.cuda}'
print('GPU:', torch.cuda.get_device_name(0))
print('Backend:', backend)
print('Torch:', torch.__version__)
print('Arquitecturas compiladas:', torch.cuda.get_arch_list())

# Prueba real: si el build no soporta tu GPU, falla acá y no a mitad del entrenamiento
a = torch.randn(1024, 1024, device=DEVICE)
b = a @ a
torch.cuda.synchronize()
print('Operación en GPU OK:', b.shape)

GPU: NVIDIA RTX 2000 Ada Generation
Backend: CUDA 12.8
Torch: 2.11.0+cu128
Arquitecturas compiladas: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
Operación en GPU OK: torch.Size([1024, 1024])


In [ ]:
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

DATA_DIR = Path(r'')   # <- cambia esto
IMG_SIZE = 64

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # todas las imágenes al mismo tamaño
    transforms.ToTensor(3, 64, 64),                     # PIL -> tensor (3, H, W), valores 0 a 1
])
dataset = datasets.ImageFolder(DATA_DIR, transform=tfm)

print('Clases:', dataset.classes)
print('Total de imágenes:', len(dataset))
conteo = Counter(dataset.targets)
for i, nombre in enumerate(dataset.classes):
    print(f'  {nombre}: {conteo[i]}')

x, y = dataset[0]
print('Forma de una imagen:', tuple(x.shape))
print('Rango de valores:', x.min().item(), 'a', x.max().item())
print('Etiqueta:', y, '->', dataset.classes[y])

# Mirar 8 imágenes para confirmar que se cargan bien
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, idx in zip(axes.flat, range(0, len(dataset), len(dataset) // 8)):
    img, label = dataset[idx]
    ax.imshow(img.permute(1, 2, 0))   # (3,H,W) -> (H,W,3) para matplotlib
    ax.set_title(dataset.classes[label], fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()